In [1]:
# Importing things I need
import scirpy as ir
import scanpy as sc
from glob import glob
import pandas as pd
import tarfile
import anndata
import warnings
import scanpy as sc
import anndata as an
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Figure 2E, 2F

In [2]:
# Importing metadata for myeloid cells from 18-144 study
mye_meta_coords = pd.read_csv('gs://fc-e9295d8f-5730-4967-b1ce-22c3d775d7c2/HC_Myeclus/mye_final_res_021326.csv', sep =',')
# print(mye_meta_coords)

# Subsetting for CXCL10 macrophage cluster
cxcl10_macs_coords = mye_meta_coords[mye_meta_coords['L2_humap_res'] == 8]
# print(cxcl10_macs_coords)

# Importing metadata for all cells from the 18-144 study
all_cells = pd.read_csv('gs://fc-e9295d8f-5730-4967-b1ce-22c3d775d7c2/data_minGenes_noDnsmpl/metadata_minGenes_noDnsmpl.txt.gz', sep='\t')
# print(len(all_cells))

# Subsetting on immune cell populations only
allowed_categories = ['B', 'Mast', 'Myeloid', 'Plasma', 'T']
all_cells = all_cells[all_cells["category"].isin(allowed_categories)].copy()
# print(len(all_cells))


In [3]:
# creating mye_coords_merged and checking counts
mye_coords_merged = mye_meta_coords
print(mye_coords_merged)

counts = (
    mye_coords_merged
        .groupby("specimenID")
        .size()
        .sort_values(ascending=False)
)

print(counts)


                    cell_id  nCount_RNA  nFeature_RNA  idx sampleID  \
0        AAAGATGCACGGCTAC-1        8072          1698    1   P02D01   
1        AAAGATGGTCGAGTTT-1        4817          1664    1   P02D01   
2        AAAGTAGGTGATAAAC-1        8405          2378    1   P02D01   
3        AAATGCCTCAGGCCCA-1        6288          2111    1   P02D01   
4        AACCATGCATCACCCT-1        6062          1813    1   P02D01   
...                     ...         ...           ...  ...      ...   
41299  TTGGCAAGTCATCGGC-103        2040           809  103   P36D15   
41300  TTTATGCCATGATCCA-103       20012          3496  103   P36D15   
41301  TTTCCTCGTCAGAAGC-103        8324          2628  103   P36D15   
41302  TTTGTCATCCAAGCCG-103       46143          4704  103   P36D15   
41303  TTTGTCATCTGATTCT-103        5754          1998  103   P36D15   

      specimenID patientID treatment tissueSite response  ...  PFSmo sex  \
0         P02D01       P02       Pre      Liver        R  ...  28.40   

In [5]:
# removing doublet clusters and remapping/numbering cluster IDs to meet specifications for the figure
mye_coords_merged_drop_doublets = mye_coords_merged[~mye_coords_merged['L2_humap_res'].isin([5, 9])].copy()

x = mye_coords_merged_drop_doublets['L2_humap_res'].astype(int)
mye_coords_merged_drop_doublets['L2_humap_res'] = np.select(
    [
        x.isin([6, 7, 8]),                 # 6,7,8 → 5,6,7
        x.isin([10, 11, 12, 13, 14]),      # 10–14 → shift down by 2
    ],
    [
        x - 1,
        x - 2,
    ],
    default=x
).astype(int)
print(len(mye_coords_merged_drop_doublets))
print(sorted(mye_coords_merged_drop_doublets['L2_humap_res'].unique().tolist()))

38707
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [4]:
# adata creation for myeloid
def merge_cell_df_into_adata_obs(
    adata,
    cell_df,
    cols=None,
    key_df=None,          # e.g. "barcode" / "cell_id" if df doesn't use index as cell id
    key_adata=None,       # e.g. "barcode" / "cell_id" if adata.obs has a column for it
    overwrite=False,
    verbose=True,):
    """
    Merge columns from a cell-level DataFrame into adata.obs.

    Best-case:
      - cell_df.index == adata.obs_names  (unique)
    Otherwise:
      - specify key_df (column in cell_df) and key_adata (column in adata.obs)
        that contain the same cell IDs.

    Returns: adata (modified in place) + a small meta dict
    """
    df = cell_df.copy()

    # choose columns to merge
    if cols is None:
        cols = list(df.columns)
    else:
        cols = [c for c in cols if c in df.columns]
        if len(cols) == 0:
            raise ValueError("None of the requested `cols` exist in cell_df.")

    # Case 1: merge by index (cell_df.index -> adata.obs_names)
    can_index_join = df.index.is_unique and pd.Index(df.index).isin(adata.obs_names).mean() > 0.50
    if key_df is None and key_adata is None and can_index_join:
        df_sub = df[cols].copy()
        # align exactly to adata.obs_names
        df_sub = df_sub.reindex(adata.obs_names)

        # avoid clobbering
        for c in cols:
            if (c in adata.obs.columns) and (not overwrite):
                if verbose:
                    print(f"[merge] skipping existing column (overwrite=False): {c}")
                df_sub = df_sub.drop(columns=[c])

        adata.obs = adata.obs.join(df_sub, how="left")

        meta = {
            "mode": "index_join",
            "n_cells_adata": int(adata.n_obs),
            "n_cells_df": int(df.shape[0]),
            "frac_df_index_in_adata": float(pd.Index(df.index).isin(adata.obs_names).mean()),
            "n_cols_added": int(df_sub.shape[1]),
        }
        if verbose:
            print("[merge] merged by index into adata.obs")
            print(meta)
        return adata, meta

    # Case 2: merge by explicit keys
    if key_df is None or key_adata is None:
        raise ValueError(
            "Could not confidently merge by index. "
            "Provide `key_df` (column in cell_df) and `key_adata` (column in adata.obs) "
            "that both contain the same cell IDs."
        )

    if key_adata not in adata.obs.columns:
        raise ValueError(f"key_adata='{key_adata}' not found in adata.obs columns.")
    if key_df not in df.columns:
        raise ValueError(f"key_df='{key_df}' not found in cell_df columns.")

    df_sub = df[[key_df] + cols].copy()
    df_sub = df_sub.drop_duplicates(subset=[key_df])

    # avoid clobbering
    cols_to_add = []
    for c in cols:
        if (c in adata.obs.columns) and (not overwrite):
            if verbose:
                print(f"[merge] skipping existing column (overwrite=False): {c}")
        else:
            cols_to_add.append(c)

    df_sub = df_sub[[key_df] + cols_to_add]

    merged = adata.obs.reset_index().rename(columns={"index": "__obsname__"})
    merged = merged.merge(df_sub, how="left", left_on=key_adata, right_on=key_df)
    merged = merged.set_index("__obsname__")
    merged = merged.drop(columns=[key_df])

    adata.obs = merged

    frac_matched = float(adata.obs[cols_to_add].notna().mean().mean()) if cols_to_add else 1.0
    meta = {
        "mode": "key_merge",
        "key_adata": key_adata,
        "key_df": key_df,
        "n_cells_adata": int(adata.n_obs),
        "n_cells_df": int(df.shape[0]),
        "n_cols_added": int(len(cols_to_add)),
        "avg_nonnull_rate_added_cols": frac_matched,
    }
    if verbose:
        print("[merge] merged by keys into adata.obs")
        print(meta)
    return adata, meta

adata_mye = sc.read_h5ad("adata_mye.h5ad")
# adata_mye, merge_meta = merge_cell_df_into_adata_obs(
#     adata=adata_mye,
#     cell_df=mye_meta_coords, # mye_coords_filtered
#     cols=['sub4_sub1_sub8_humap_fgraph_res'], # [cluster_col]
#     key_df="cell_id",        # <-- change
#     key_adata="Unnamed: 0",     # <-- change
#     overwrite=True,
#     verbose=True,
# )

cols_to_merge = [
    "L2_humap_res",
    "HUMAP_1",
    "HUMAP_2",
]

adata_mye, merge_meta = merge_cell_df_into_adata_obs(
    adata=adata_mye,
    cell_df=mye_coords_merged, #_drop_doublets,
    cols=cols_to_merge,  # <-- list, not string
    key_df="cell_id",
    key_adata="Unnamed: 0",
    overwrite=True,
    verbose=True,
)



[merge] merged by keys into adata.obs
{'mode': 'key_merge', 'key_adata': 'Unnamed: 0', 'key_df': 'cell_id', 'n_cells_adata': 41304, 'n_cells_df': 41304, 'n_cols_added': 3, 'avg_nonnull_rate_added_cols': 1.0}


In [6]:
# subset adata to only the cells present in mye_coords_merged_drop_doublets
cells_keep = pd.Index(mye_coords_merged_drop_doublets["cell_id"].astype(str).unique())

# if adata obs key isn't string, normalized
adata_keys = adata_mye.obs["Unnamed: 0"].astype(str)

mask = adata_keys.isin(cells_keep).values
adata_mye = adata_mye[mask].copy()

print("After subsetting:", adata_mye.n_obs, "cells")
print("Expected (unique df cells):", len(cells_keep))


After subsetting: 38707 cells
Expected (unique df cells): 38707


In [11]:
# Compute gene signature score activity per cell using Scanpy gene-set scoring, then pseudobulk scores to the specimen 
# level (mean/median across cells). Paired D01 vs D15 changes are calculated per patient, and significance is 
# tested using a paired Wilcoxon test overall and by PFS > 6 months or < 6 months group with BH-FDR correction.
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

def _get_X_and_genes(adata, use_raw=False, layer=None):
    if use_raw and getattr(adata, "raw", None) is not None:
        X = adata.raw.X
        var_names = adata.raw.var_names
    else:
        X = adata.layers[layer] if layer is not None else adata.X
        var_names = adata.var_names
    return X, var_names


def _score_genes_per_cell_scanpy(
    adata,
    genes,
    score_key="MAPK_score",
    use_raw=False,
    layer=None,
    ctrl_size=50,
    n_bins=25,
    random_state=0,):
    """
    Uses scanpy.sc.tl.score_genes to compute a per-cell gene-set score.
    Returns (scores_1d, genes_present, adata_scored).
    """
    ad = adata.copy()

    # If a layer is requested, score on that layer by temporarily swapping X.
    # (scanpy's score_genes historically scores on .X; use_raw overrides.)
    if layer is not None:
        if use_raw:
            raise ValueError("If use_raw=True, please do not pass layer (score_genes will use .raw).")
        if layer not in ad.layers:
            raise ValueError(f"layer='{layer}' not found in adata.layers")
        ad.X = ad.layers[layer]

    # Determine which gene names we are matching against
    if use_raw and getattr(ad, "raw", None) is not None:
        var_names = np.asarray(ad.raw.var_names)
        use_raw_arg = True
    else:
        var_names = np.asarray(ad.var_names)
        use_raw_arg = False

    gene_set = set(var_names.tolist())
    genes_present = [g for g in genes if g in gene_set]
    if len(genes_present) == 0:
        raise ValueError("None of the provided genes were found in adata.var_names (or adata.raw.var_names).")

    # Score genes
    sc.tl.score_genes(
        ad,
        gene_list=genes_present,
        score_name=score_key,
        ctrl_size=ctrl_size,
        n_bins=n_bins,
        random_state=random_state,
        use_raw=use_raw_arg,
        copy=False,
    )

    scores = ad.obs[score_key].to_numpy(dtype=float)
    return scores, genes_present, ad


def mapk_change_stats_from_adata(
    adata_mye,
    genes,
    specimen_col="specimenID",
    cluster_col="L2_humap_res",
    exclude_clusters=None,                 # filter OUT these clusters first
    patient_parser=lambda s: s.str.slice(0, 3),
    day_parser=lambda s: s.str.slice(3, 6),
    d01="D01",
    d15="D15",
    nonresponder_patients=None,
    responder_patients=None,
    exclude_patients=None,                   # e.g. ["P15","P23"]
    use_raw=False,
    layer=None,
    pseudobulk_agg="median",                   # "mean" or "median" per specimen
    dropna_pairs=True,
    run_overall=True,
    run_by_group=True,
    fdr_method="fdr_bh",
    # score_genes params
    ctrl_size=50,
    n_bins=25,
    random_state=0,):
    """
    Computes MAPK score per cell via sc.tl.score_genes, then pseudobulks to 1 value per specimen,
    then tests paired change (D15 - D01) per patient using Wilcoxon, with BH-FDR.

    Returns:
      specimen_tbl: one row per specimenID with pseudobulk MAPK score (+ patient/day)
      wide_tbl: per-patient D01/D15 MAPK scores + delta + group
      tests: overall + by-group Wilcoxon, with BH-adjusted q-values
      meta: dict with genes_present, n_cells_used, etc.
    """
    ad = adata_mye.copy()

    # --- filter out clusters (e.g., 5 and 9) ---
    if cluster_col is not None and exclude_clusters is not None:
        if cluster_col not in ad.obs:
            raise ValueError(f"cluster_col='{cluster_col}' not found in adata.obs")
        ad = ad[~ad.obs[cluster_col].isin(list(exclude_clusters))].copy()

    # --- require specimen_col ---
    if specimen_col not in ad.obs:
        raise ValueError(f"specimen_col='{specimen_col}' not found in adata.obs")

    # --- compute per-cell MAPK score using scanpy ---
    per_cell_score, genes_present, ad_scored = _score_genes_per_cell_scanpy(
        ad,
        genes=genes,
        score_key="MAPK_score",
        use_raw=use_raw,
        layer=layer,
        ctrl_size=ctrl_size,
        n_bins=n_bins,
        random_state=random_state,
    )

    # --- specimen-level pseudobulk ---
    obs = ad_scored.obs[[specimen_col, "MAPK_score"]].copy()
    obs["patient"] = patient_parser(obs[specimen_col].astype(str))
    obs["day"] = day_parser(obs[specimen_col].astype(str))

    if exclude_patients is not None:
        obs = obs[~obs["patient"].isin(list(exclude_patients))].copy()

    if pseudobulk_agg == "mean":
        specimen_tbl = (obs.groupby(specimen_col)
                          .agg(MAPK_score=("MAPK_score", "mean"),
                               patient=("patient", "first"),
                               day=("day", "first"),
                               n_cells=("MAPK_score", "size"))
                          .reset_index())
    elif pseudobulk_agg == "median":
        specimen_tbl = (obs.groupby(specimen_col)
                          .agg(MAPK_score=("MAPK_score", "median"),
                               patient=("patient", "first"),
                               day=("day", "first"),
                               n_cells=("MAPK_score", "size"))
                          .reset_index())
    else:
        raise ValueError("pseudobulk_agg must be 'mean' or 'median'")

    # --- patient x day wide table ---
    wide_tbl = (specimen_tbl.pivot_table(index="patient", columns="day", values="MAPK_score", aggfunc="first")
                            .rename(columns={d01: f"{d01}_MAPK", d15: f"{d15}_MAPK"})
                            .reset_index())

    if dropna_pairs:
        wide_tbl = wide_tbl.dropna(subset=[f"{d01}_MAPK", f"{d15}_MAPK"]).copy()

    wide_tbl["delta"] = wide_tbl[f"{d15}_MAPK"] - wide_tbl[f"{d01}_MAPK"]

    # --- label responder status ---
    wide_tbl["group"] = "unknown"
    if nonresponder_patients is not None:
        wide_tbl.loc[wide_tbl["patient"].isin(nonresponder_patients), "group"] = "nonresponder"
        wide_tbl.loc[~wide_tbl["patient"].isin(nonresponder_patients), "group"] = "responder"
    elif responder_patients is not None:
        wide_tbl.loc[wide_tbl["patient"].isin(responder_patients), "group"] = "responder"
        wide_tbl.loc[~wide_tbl["patient"].isin(responder_patients), "group"] = "nonresponder"

    # --- paired Wilcoxon on delta vs 0 ---
    def _wilcox_delta(delta_series, label):
        d = delta_series.dropna().astype(float).values
        if len(d) < 3:
            return {"test": label, "n": len(d), "stat": np.nan, "p": np.nan}
        if np.allclose(d, 0):
            return {"test": label, "n": len(d), "stat": 0.0, "p": 1.0}
        try:
            out = wilcoxon(d, zero_method="wilcox", alternative="two-sided")
            return {"test": label, "n": len(d), "stat": float(out.statistic), "p": float(out.pvalue)}
        except Exception:
            return {"test": label, "n": len(d), "stat": np.nan, "p": np.nan}

    test_rows = []
    if run_overall:
        test_rows.append(_wilcox_delta(wide_tbl["delta"], "overall: D15-D01 MAPK vs 0"))
    if run_by_group:
        for g in ["responder", "nonresponder"]:
            test_rows.append(
                _wilcox_delta(
                    wide_tbl.loc[wide_tbl["group"] == g, "delta"],
                    f"{g}: D15-D01 MAPK vs 0"
                )
            )

    tests = pd.DataFrame(test_rows)

    # --- BH-FDR across the tests we ran ---
    if tests["p"].notna().any():
        pvals = tests["p"].fillna(1.0).values
        tests["q"] = multipletests(pvals, method=fdr_method)[1]
        tests["fdr_method"] = fdr_method
    else:
        tests["q"] = np.nan
        tests["fdr_method"] = fdr_method

    meta = {
        "genes_requested": list(genes),
        "genes_present": genes_present,
        "n_genes_present": len(genes_present),
        "exclude_clusters": list(exclude_clusters) if exclude_clusters is not None else None,
        "n_cells_used": int(ad_scored.n_obs),
        "pseudobulk_agg": pseudobulk_agg,
        "use_raw": use_raw,
        "layer": layer,
        "score_method": "scanpy.tl.score_genes",
        "ctrl_size": ctrl_size,
        "n_bins": n_bins,
        "random_state": random_state,
    }

    return specimen_tbl, wide_tbl, tests, meta


In [9]:
# Running functions detailed above on MAPK genes (9 gene score)
# Nonresponders (PFS < 6 months) are below can interchange with Responders (PFS > 6 months)
MAPK_GENES = ["SPRY2", "SPRY4", "ETV4", "ETV5", "DUSP4", "DUSP6", "CCND1", "EPHA2", "EPHA4"]
nonresponders = ["P03","P14",'P15',"P19","P25","P27","P28","P29","P30","P33","P35","P36"]

MAPK_specimen_tbl, MAPK_wide_tbl, MAPK_test_tbl, MAPK_meta = mapk_change_stats_from_adata(
    adata_mye=adata_mye,
    genes=MAPK_GENES,
    cluster_col="L2_humap_res",
    exclude_clusters=None,
    nonresponder_patients=nonresponders,
    exclude_patients= None,
    pseudobulk_agg="median",
    fdr_method="fdr_bh",
    use_raw=False,
    layer=None,
)
print(MAPK_meta)
print(MAPK_specimen_tbl)  # one row per specimenID (pseudobulked MAPK)
print(MAPK_wide_tbl)      # one row per patient with D01/D15 + delta
print(MAPK_test_tbl)             # Wilcoxon + BH q-values
MAPK_wide_tbl.to_csv('021726_myeloid_MAPK_scores.csv', index=False)

{'genes_requested': ['SPRY2', 'SPRY4', 'ETV4', 'ETV5', 'DUSP4', 'DUSP6', 'CCND1', 'EPHA2', 'EPHA4'], 'genes_present': ['SPRY2', 'SPRY4', 'ETV4', 'ETV5', 'DUSP4', 'DUSP6', 'CCND1', 'EPHA2', 'EPHA4'], 'n_genes_present': 9, 'exclude_clusters': None, 'n_cells_used': 38707, 'pseudobulk_agg': 'median', 'use_raw': False, 'layer': None, 'score_method': 'scanpy.tl.score_genes', 'ctrl_size': 50, 'n_bins': 25, 'random_state': 0}
   specimenID  MAPK_score patient  day  n_cells
0      P02D01   -0.023385     P02  D01      196
1      P02D15   -0.081614     P02  D15      147
2      P03D01    0.077058     P03  D01       42
3      P03D15   -0.089863     P03  D15       44
4      P07D01   -0.023708     P07  D01      356
5      P10D01   -0.070680     P10  D01     1452
6      P10D15   -0.102968     P10  D15     1528
7      P11D01    0.029979     P11  D01      303
8      P11D15   -0.080179     P11  D15      122
9      P12D01   -0.061342     P12  D01      445
10     P13D01   -0.125901     P13  D01      711
11

In [10]:
# Running functions detailed above on pan-ISG score
# Nonresponders (PFS < 6 months) are below can interchange with Responders (PFS > 6 months)

ISG_GENES = ["HLA-DRB1","CD74","PSMB9","HLA-DQB1","HLA-DRA","HLA-B","PSMB10","HLA-DPA1",
    "B2M","HLA-F","PSME2","HLA-C","TAP2","TAP1","HLA-A","HLA-DRB5","HLA-DQA2",
    "CIITA","HLA-DQB2","HLA-DPB1","NOS2","GBP7","CCL8","GBP4","TRIM22","CASP1",
    "UBD","GBP1","IFNG","CCL23","XCL1","XCL2","STAT1","BST2","CX3CL1","TLR2",
    "GBP5","DAPK1","CCL3","CCL4","CCL20","IRF1","CCL5","SIRPA","IFI30","GBP2",
    "PARP9","IL12RB1","NR1H3","TRIM31","IRF9","IFITM3","GBP3","PARP14","CCL3L1",
    "EVL","SOCS1","TRIM21","IFITM1","STAT2","ZBP1","C19orf66","IFI35","IFIT2",
    "IFIT3","ISG15","IFNAR2","XAF1","IFI27","MX1","IFIT1","RSAD2","IFI6",
    "C10orf99","CXCL10","CXCL11","CXCL9","CXCL13","CXCL5","CXCL6"]

nonresponders = ["P03","P14",'P15',"P19","P25","P27","P28","P29","P30","P33","P35","P36"]

ISG_specimen_tbl, ISG_wide_tbl, ISG_test_tbl, ISG_meta = mapk_change_stats_from_adata(
    adata_mye=adata_mye,
    genes=ISG_GENES,
    cluster_col="L2_humap_res",
    exclude_clusters=None,
    nonresponder_patients=nonresponders,
    exclude_patients= None,
    pseudobulk_agg="median",
    fdr_method="fdr_bh",
    use_raw=False,
    layer=None,
)
print(ISG_meta)
print(ISG_specimen_tbl)  # one row per specimenID (pseudobulked ISG)
print(ISG_wide_tbl)      # one row per patient with D01/D15 + delta
print(ISG_test_tbl)             # Wilcoxon + BH q-values
ISG_wide_tbl.to_csv('021726_myeloid_ISG_scores.csv', index=False)

{'genes_requested': ['HLA-DRB1', 'CD74', 'PSMB9', 'HLA-DQB1', 'HLA-DRA', 'HLA-B', 'PSMB10', 'HLA-DPA1', 'B2M', 'HLA-F', 'PSME2', 'HLA-C', 'TAP2', 'TAP1', 'HLA-A', 'HLA-DRB5', 'HLA-DQA2', 'CIITA', 'HLA-DQB2', 'HLA-DPB1', 'NOS2', 'GBP7', 'CCL8', 'GBP4', 'TRIM22', 'CASP1', 'UBD', 'GBP1', 'IFNG', 'CCL23', 'XCL1', 'XCL2', 'STAT1', 'BST2', 'CX3CL1', 'TLR2', 'GBP5', 'DAPK1', 'CCL3', 'CCL4', 'CCL20', 'IRF1', 'CCL5', 'SIRPA', 'IFI30', 'GBP2', 'PARP9', 'IL12RB1', 'NR1H3', 'TRIM31', 'IRF9', 'IFITM3', 'GBP3', 'PARP14', 'CCL3L1', 'EVL', 'SOCS1', 'TRIM21', 'IFITM1', 'STAT2', 'ZBP1', 'C19orf66', 'IFI35', 'IFIT2', 'IFIT3', 'ISG15', 'IFNAR2', 'XAF1', 'IFI27', 'MX1', 'IFIT1', 'RSAD2', 'IFI6', 'C10orf99', 'CXCL10', 'CXCL11', 'CXCL9', 'CXCL13', 'CXCL5', 'CXCL6'], 'genes_present': ['HLA-DRB1', 'CD74', 'PSMB9', 'HLA-DQB1', 'HLA-DRA', 'HLA-B', 'PSMB10', 'HLA-DPA1', 'B2M', 'HLA-F', 'PSME2', 'HLA-C', 'TAP2', 'TAP1', 'HLA-A', 'HLA-DRB5', 'HLA-DQA2', 'CIITA', 'HLA-DQB2', 'HLA-DPB1', 'NOS2', 'GBP7', 'CCL8', 'GBP4

In [12]:
# Compute gene signature score activity per cell on a per-cluster basis for the pan-ISG score.
# Question: are certain clusters responsible for significant shift in ISG score on treatment? If so, which ones?
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

def _get_X_and_genes(adata, use_raw=False, layer=None):
    if use_raw and getattr(adata, "raw", None) is not None:
        X = adata.raw.X
        var_names = adata.raw.var_names
    else:
        X = adata.layers[layer] if layer is not None else adata.X
        var_names = adata.var_names
    return X, var_names

def _score_genes_per_cell_scanpy(
    adata,
    genes,
    score_key="MAPK_score",
    use_raw=False,
    layer=None,
    ctrl_size=50,
    n_bins=25,
    random_state=0,):
    """
    Uses scanpy.sc.tl.score_genes to compute a per-cell gene-set score.
    Returns (scores_1d, genes_present, adata_scored).
    """
    import scanpy as sc

    ad = adata.copy()

    # If a layer is requested, score on that layer by temporarily swapping X.
    if layer is not None:
        if use_raw:
            raise ValueError("If use_raw=True, please do not pass layer (score_genes will use .raw).")
        if layer not in ad.layers:
            raise ValueError(f"layer='{layer}' not found in adata.layers")
        ad.X = ad.layers[layer]

    # Determine which gene names we are matching against
    if use_raw and getattr(ad, "raw", None) is not None:
        var_names = np.asarray(ad.raw.var_names)
        use_raw_arg = True
    else:
        var_names = np.asarray(ad.var_names)
        use_raw_arg = False

    gene_set = set(var_names.tolist())
    genes_present = [g for g in genes if g in gene_set]
    if len(genes_present) == 0:
        raise ValueError("None of the provided genes were found in adata.var_names (or adata.raw.var_names).")

    sc.tl.score_genes(
        ad,
        gene_list=genes_present,
        score_name=score_key,
        ctrl_size=ctrl_size,
        n_bins=n_bins,
        random_state=random_state,
        use_raw=use_raw_arg,
        copy=False,
    )

    scores = ad.obs[score_key].to_numpy(dtype=float)
    return scores, genes_present, ad

def mapk_change_stats_from_adata(
    adata_mye,
    genes,
    specimen_col="specimenID",
    cluster_col="L2_humap_res",
    exclude_clusters=None,
    patient_parser=lambda s: s.str.slice(0, 3),
    day_parser=lambda s: s.str.slice(3, 6),
    d01="D01",
    d15="D15",
    nonresponder_patients=None,
    responder_patients=None,
    exclude_patients=None,
    use_raw=False,
    layer=None,
    pseudobulk_agg="median",
    dropna_pairs=True,
    run_overall=True,
    run_by_group=True,
    fdr_method="fdr_bh",
    # per-cluster testing
    run_cluster_tests=True,
    cluster_test_alternative="greater",
    cluster_fdr_scope="all",
    min_pairs_per_test=3,
    # NEW: minimum cells per patient×cluster×timepoint
    min_cells_per_patient_cluster=3,
    # score_genes params
    ctrl_size=50,
    n_bins=25,
    random_state=0,):
    """
    Same as your function, but per-cluster tests enforce:
      For a given patient×cluster to be included, both D01 and D15 must have
      >= min_cells_per_patient_cluster cells in that cluster (per timepoint).
    """

    # --------------------------
    # Helper: score genes per cell
    # --------------------------
    def _score_genes_per_cell_scanpy(
        adata,
        genes,
        score_key="MAPK_score",
        use_raw=False,
        layer=None,
        ctrl_size=50,
        n_bins=25,
        random_state=0,
    ):
        import scanpy as sc

        ad = adata.copy()

        if layer is not None:
            if use_raw:
                raise ValueError("If use_raw=True, do not pass layer (score_genes will use .raw).")
            if layer not in ad.layers:
                raise ValueError(f"layer='{layer}' not found in adata.layers")
            ad.X = ad.layers[layer]

        if use_raw and getattr(ad, "raw", None) is not None:
            var_names = np.asarray(ad.raw.var_names)
            use_raw_arg = True
        else:
            var_names = np.asarray(ad.var_names)
            use_raw_arg = False

        gene_set = set(var_names.tolist())
        genes_present = [g for g in genes if g in gene_set]
        if len(genes_present) == 0:
            raise ValueError("None of the provided genes were found in var_names/raw.var_names.")

        sc.tl.score_genes(
            ad,
            gene_list=genes_present,
            score_name=score_key,
            ctrl_size=ctrl_size,
            n_bins=n_bins,
            random_state=random_state,
            use_raw=use_raw_arg,
            copy=False,
        )
        scores = ad.obs[score_key].to_numpy(dtype=float)
        return scores, genes_present, ad

    # --------------------------
    # Helper: wilcoxon on deltas
    # --------------------------
    def _wilcox_delta(delta_series, label, alternative="two-sided"):
        d = pd.Series(delta_series).dropna().astype(float).values
        if len(d) < min_pairs_per_test:
            return {
                "test": label,
                "n": len(d),
                "stat": np.nan,
                "p": np.nan,
                "alternative": alternative,
                "median_delta": float(np.nanmedian(d)) if len(d) else np.nan,
            }
        if np.allclose(d, 0):
            return {
                "test": label,
                "n": len(d),
                "stat": 0.0,
                "p": 1.0,
                "alternative": alternative,
                "median_delta": float(np.median(d)),
            }
        out = wilcoxon(d, zero_method="wilcox", alternative=alternative)
        return {
            "test": label,
            "n": len(d),
            "stat": float(out.statistic),
            "p": float(out.pvalue),
            "alternative": alternative,
            "median_delta": float(np.median(d)),
        }

    # --------------------------
    # Start
    # --------------------------
    ad = adata_mye.copy()

    # filter out clusters globally (e.g., 5 and 9)
    if cluster_col is not None and exclude_clusters is not None:
        if cluster_col not in ad.obs:
            raise ValueError(f"cluster_col='{cluster_col}' not found in adata.obs")
        ad = ad[~ad.obs[cluster_col].isin(list(exclude_clusters))].copy()

    if specimen_col not in ad.obs:
        raise ValueError(f"specimen_col='{specimen_col}' not found in adata.obs")

    # score per cell
    _, genes_present, ad_scored = _score_genes_per_cell_scanpy(
        ad,
        genes=genes,
        score_key="MAPK_score",
        use_raw=use_raw,
        layer=layer,
        ctrl_size=ctrl_size,
        n_bins=n_bins,
        random_state=random_state,
    )

    # build obs table we use for pseudobulking
    obs = ad_scored.obs[[specimen_col, "MAPK_score", cluster_col]].copy()
    obs["patient"] = patient_parser(obs[specimen_col].astype(str))
    obs["day"] = day_parser(obs[specimen_col].astype(str))

    if exclude_patients is not None:
        obs = obs[~obs["patient"].isin(list(exclude_patients))].copy()

    # =========================
    # Overall specimen pseudobulk (unchanged behavior)
    # =========================
    agg_func = "mean" if pseudobulk_agg == "mean" else "median"
    specimen_tbl = (
        obs.groupby(specimen_col)
        .agg(
            MAPK_score=("MAPK_score", agg_func),
            patient=("patient", "first"),
            day=("day", "first"),
            n_cells=("MAPK_score", "size"),
        )
        .reset_index()
    )

    wide_tbl = (
        specimen_tbl.pivot_table(index="patient", columns="day", values="MAPK_score", aggfunc="first")
        .rename(columns={d01: f"{d01}_MAPK", d15: f"{d15}_MAPK"})
        .reset_index()
    )
    if dropna_pairs:
        wide_tbl = wide_tbl.dropna(subset=[f"{d01}_MAPK", f"{d15}_MAPK"]).copy()
    wide_tbl["delta"] = wide_tbl[f"{d15}_MAPK"] - wide_tbl[f"{d01}_MAPK"]

    wide_tbl["group"] = "unknown"
    if nonresponder_patients is not None:
        wide_tbl.loc[wide_tbl["patient"].isin(nonresponder_patients), "group"] = "nonresponder"
        wide_tbl.loc[~wide_tbl["patient"].isin(nonresponder_patients), "group"] = "responder"
    elif responder_patients is not None:
        wide_tbl.loc[wide_tbl["patient"].isin(responder_patients), "group"] = "responder"
        wide_tbl.loc[~wide_tbl["patient"].isin(responder_patients), "group"] = "nonresponder"

    test_rows = []
    if run_overall:
        test_rows.append(_wilcox_delta(wide_tbl["delta"], "overall: D15-D01 MAPK vs 0", alternative="two-sided"))
    if run_by_group:
        for g in ["responder", "nonresponder"]:
            test_rows.append(
                _wilcox_delta(
                    wide_tbl.loc[wide_tbl["group"] == g, "delta"],
                    f"{g}: D15-D01 MAPK vs 0",
                    alternative="two-sided",
                )
            )
    tests = pd.DataFrame(test_rows)
    if tests["p"].notna().any():
        tests["q"] = multipletests(tests["p"].fillna(1.0).values, method=fdr_method)[1]
        tests["fdr_method"] = fdr_method
    else:
        tests["q"] = np.nan
        tests["fdr_method"] = fdr_method

    # =========================
    # Per-cluster tests with min cells per patient×cluster×day
    # =========================
    cluster_tests = pd.DataFrame()
    cluster_wide = pd.DataFrame()

    if run_cluster_tests:
        # pseudobulk per specimen×cluster, keep n_cells
        cl_spec = (
            obs.groupby([specimen_col, cluster_col])
            .agg(
                score=("MAPK_score", agg_func),
                patient=("patient", "first"),
                day=("day", "first"),
                n_cells=("MAPK_score", "size"),
            )
            .reset_index()
            .rename(columns={cluster_col: "cluster"})
        )

        # Pivot scores AND n_cells to wide (patient×cluster)
        score_wide = (
            cl_spec.pivot_table(index=["patient", "cluster"], columns="day", values="score", aggfunc="first")
            .rename(columns={d01: f"{d01}_score", d15: f"{d15}_score"})
            .reset_index()
        )
        ncell_wide = (
            cl_spec.pivot_table(index=["patient", "cluster"], columns="day", values="n_cells", aggfunc="first")
            .rename(columns={d01: f"{d01}_n_cells", d15: f"{d15}_n_cells"})
            .reset_index()
        )

        cluster_wide = score_wide.merge(ncell_wide, on=["patient", "cluster"], how="left")

        # label group
        cluster_wide["group"] = "unknown"
        if nonresponder_patients is not None:
            cluster_wide.loc[cluster_wide["patient"].isin(nonresponder_patients), "group"] = "nonresponder"
            cluster_wide.loc[~cluster_wide["patient"].isin(nonresponder_patients), "group"] = "responder"
        elif responder_patients is not None:
            cluster_wide.loc[cluster_wide["patient"].isin(responder_patients), "group"] = "responder"
            cluster_wide.loc[~cluster_wide["patient"].isin(responder_patients), "group"] = "nonresponder"

        # REQUIRE: both timepoints present AND each has >= min cells
        req_cols = [f"{d01}_score", f"{d15}_score", f"{d01}_n_cells", f"{d15}_n_cells"]
        if dropna_pairs:
            cluster_wide = cluster_wide.dropna(subset=req_cols).copy()
        else:
            # even if not dropna_pairs, we cannot test without both scores
            cluster_wide = cluster_wide.dropna(subset=[f"{d01}_score", f"{d15}_score"]).copy()

        cluster_wide = cluster_wide[
            (cluster_wide[f"{d01}_n_cells"] >= min_cells_per_patient_cluster) &
            (cluster_wide[f"{d15}_n_cells"] >= min_cells_per_patient_cluster)
        ].copy()

        # compute deltas
        cluster_wide["delta"] = cluster_wide[f"{d15}_score"] - cluster_wide[f"{d01}_score"]

        # run wilcoxon per cluster overall and per group
        rows = []
        for cl in sorted(cluster_wide["cluster"].unique()):
            sub = cluster_wide.loc[cluster_wide["cluster"] == cl]

            rows.append(
                (_wilcox_delta(
                    sub["delta"],
                    label=f"cluster {cl} overall: D15-D01 score > 0",
                    alternative=cluster_test_alternative,
                ) | {"cluster": cl, "group": "overall"})
            )

            if run_by_group:
                for g in ["responder", "nonresponder"]:
                    rows.append(
                        (_wilcox_delta(
                            sub.loc[sub["group"] == g, "delta"],
                            label=f"cluster {cl} {g}: D15-D01 score > 0",
                            alternative=cluster_test_alternative,
                        ) | {"cluster": cl, "group": g})
                    )

        cluster_tests = pd.DataFrame(rows)

        # FDR correction
        if cluster_tests["p"].notna().any():
            if cluster_fdr_scope == "within_cluster":
                cluster_tests["q"] = np.nan
                for cl in cluster_tests["cluster"].unique():
                    mask = cluster_tests["cluster"] == cl
                    cluster_tests.loc[mask, "q"] = multipletests(
                        cluster_tests.loc[mask, "p"].fillna(1.0).values,
                        method=fdr_method
                    )[1]
            else:
                cluster_tests["q"] = multipletests(cluster_tests["p"].fillna(1.0).values, method=fdr_method)[1]
            cluster_tests["fdr_method"] = fdr_method
            cluster_tests["fdr_scope"] = cluster_fdr_scope
        else:
            cluster_tests["q"] = np.nan
            cluster_tests["fdr_method"] = fdr_method
            cluster_tests["fdr_scope"] = cluster_fdr_scope

        cluster_tests["significant_increase_q05"] = (cluster_tests["q"] < 0.05) & (cluster_tests["median_delta"] > 0)

        cluster_tests = (
            cluster_tests[
                ["cluster", "group", "n", "median_delta", "stat", "p", "q",
                 "significant_increase_q05", "alternative", "fdr_method", "fdr_scope"]
            ]
            .sort_values(["cluster", "group"])
            .reset_index(drop=True)
        )

    meta = {
        "genes_requested": list(genes),
        "genes_present": genes_present,
        "n_genes_present": len(genes_present),
        "exclude_clusters": list(exclude_clusters) if exclude_clusters is not None else None,
        "n_cells_used": int(ad_scored.n_obs),
        "pseudobulk_agg": pseudobulk_agg,
        "use_raw": use_raw,
        "layer": layer,
        "score_method": "scanpy.tl.score_genes",
        "ctrl_size": ctrl_size,
        "n_bins": n_bins,
        "random_state": random_state,
        "cluster_tests": bool(run_cluster_tests),
        "cluster_test_alternative": cluster_test_alternative,
        "cluster_fdr_scope": cluster_fdr_scope,
        "min_pairs_per_test": min_pairs_per_test,
        "min_cells_per_patient_cluster": min_cells_per_patient_cluster,
    }

    return specimen_tbl, wide_tbl, tests, meta, cluster_tests, cluster_wide


In [13]:
# Running functions detailed above on pan-ISG score on per-cluster basis, outputs saved
ISG_GENES = ["HLA-DRB1","CD74","PSMB9","HLA-DQB1","HLA-DRA","HLA-B","PSMB10","HLA-DPA1",
             "B2M","HLA-F","PSME2","HLA-C","TAP2","TAP1","HLA-A","HLA-DRB5","HLA-DQA2",
             "CIITA","HLA-DQB2","HLA-DPB1","NOS2","GBP7","CCL8","GBP4","TRIM22","CASP1",
             "UBD","GBP1","IFNG","CCL23","XCL1","XCL2","STAT1","BST2","CX3CL1","TLR2",
             "GBP5","DAPK1","CCL3","CCL4","CCL20","IRF1","CCL5","SIRPA","IFI30","GBP2",
             "PARP9","IL12RB1","NR1H3","TRIM31","IRF9","IFITM3","GBP3","PARP14","CCL3L1",
             "EVL","SOCS1","TRIM21","IFITM1","STAT2","ZBP1","C19orf66","IFI35","IFIT2",
             "IFIT3","ISG15","IFNAR2","XAF1","IFI27","MX1","IFIT1","RSAD2","IFI6",
             "C10orf99","CXCL10","CXCL11","CXCL9","CXCL13","CXCL5","CXCL6"]

nonresponders = ["P03","P14","P15","P19","P25","P27","P28","P29","P30","P33","P35","P36"]

ISG_specimen_tbl, ISG_wide_tbl, ISG_test_tbl, ISG_meta, ISG_cluster_tests, ISG_cluster_wide = mapk_change_stats_from_adata(
    adata_mye=adata_mye,
    genes=ISG_GENES,
    cluster_col="L2_humap_res",
    exclude_clusters=None,
    nonresponder_patients=nonresponders,
    exclude_patients=None,
    pseudobulk_agg="median",
    fdr_method="fdr_bh",
    use_raw=False,
    layer=None,
    run_cluster_tests=True,
    cluster_test_alternative="greater",   # <-- tests for increase
    cluster_fdr_scope="all",              # BH across all cluster+group tests
)

print(ISG_meta)

# Existing outputs
print(ISG_specimen_tbl.head())
print(ISG_wide_tbl.head())
print(ISG_test_tbl)

# NEW: this is the table you asked for (responder/nonresponder per cluster)
print(ISG_cluster_tests)

# Save what you want
ISG_wide_tbl.to_csv("021726_myeloid_ISG_scores.csv", index=False)
ISG_cluster_tests.to_csv("021726_myeloid_ISG_cluster_increase_tests.csv", index=False)
ISG_cluster_wide.to_csv("021726_myeloid_ISG_cluster_patient_wide.csv", index=False)


{'genes_requested': ['HLA-DRB1', 'CD74', 'PSMB9', 'HLA-DQB1', 'HLA-DRA', 'HLA-B', 'PSMB10', 'HLA-DPA1', 'B2M', 'HLA-F', 'PSME2', 'HLA-C', 'TAP2', 'TAP1', 'HLA-A', 'HLA-DRB5', 'HLA-DQA2', 'CIITA', 'HLA-DQB2', 'HLA-DPB1', 'NOS2', 'GBP7', 'CCL8', 'GBP4', 'TRIM22', 'CASP1', 'UBD', 'GBP1', 'IFNG', 'CCL23', 'XCL1', 'XCL2', 'STAT1', 'BST2', 'CX3CL1', 'TLR2', 'GBP5', 'DAPK1', 'CCL3', 'CCL4', 'CCL20', 'IRF1', 'CCL5', 'SIRPA', 'IFI30', 'GBP2', 'PARP9', 'IL12RB1', 'NR1H3', 'TRIM31', 'IRF9', 'IFITM3', 'GBP3', 'PARP14', 'CCL3L1', 'EVL', 'SOCS1', 'TRIM21', 'IFITM1', 'STAT2', 'ZBP1', 'C19orf66', 'IFI35', 'IFIT2', 'IFIT3', 'ISG15', 'IFNAR2', 'XAF1', 'IFI27', 'MX1', 'IFIT1', 'RSAD2', 'IFI6', 'C10orf99', 'CXCL10', 'CXCL11', 'CXCL9', 'CXCL13', 'CXCL5', 'CXCL6'], 'genes_present': ['HLA-DRB1', 'CD74', 'PSMB9', 'HLA-DQB1', 'HLA-DRA', 'HLA-B', 'PSMB10', 'HLA-DPA1', 'B2M', 'HLA-F', 'PSME2', 'HLA-C', 'TAP2', 'TAP1', 'HLA-A', 'HLA-DRB5', 'HLA-DQA2', 'CIITA', 'HLA-DQB2', 'HLA-DPB1', 'NOS2', 'GBP7', 'CCL8', 'GBP4

In [25]:
def clean_and_merge_data(adata, t_meta_coords):
    adata_cleaned = adata.copy()
    adata_cleaned.obs.rename(columns={'Unnamed: 0': 'cell_id'}, inplace=True)
    adata_cleaned.obs = pd.merge(adata_cleaned.obs, t_meta_coords, how='left', on=['cell_id', 'specimenID'])
    adata_cleaned.obs = adata_cleaned.obs.loc[:, ~adata_cleaned.obs.columns.duplicated()]
    
    suffixes_to_drop = ['_x', '_y']
    for suffix in suffixes_to_drop:
        columns_to_drop = [col for col in adata_cleaned.obs.columns if col.endswith(suffix) and col.rstrip(suffix) in adata_cleaned.obs.columns]
        adata_cleaned.obs.drop(columns=columns_to_drop, inplace=True)
    
    return adata_cleaned

mye_clus = pd.read_csv('gs://fc-e9295d8f-5730-4967-b1ce-22c3d775d7c2/HC_Myeclus/mye_annotated_060724.csv', sep = ',')


In [26]:
adata_mye_g6 = adata_mye[adata_mye.obs['PFSmo'] >= 6.0]
adata_mye_l6 = adata_mye[adata_mye.obs['PFSmo'] < 6]
adata_mye_on = adata_mye[adata_mye.obs['treatment'] == 'On']
adata_mye_pre = adata_mye[adata_mye.obs['treatment'] == 'Pre']
adata_mye_g6_on = adata_mye_on[adata_mye_on.obs['PFSmo'] >= 6.0]
adata_mye_g6_pre = adata_mye_pre[adata_mye_pre.obs['PFSmo'] >= 6.0]
adata_mye_l6_on = adata_mye_on[adata_mye_on.obs['PFSmo'] < 6.0 ]
adata_mye_l6_pre = adata_mye_pre[adata_mye_pre.obs['PFSmo'] < 6.0]

In [27]:
data_mye_g6_on = clean_and_merge_data(adata_mye_g6_on, mye_clus)
data_mye_g6_pre = clean_and_merge_data(adata_mye_g6_pre, mye_clus)
data_mye_l6_on = clean_and_merge_data(adata_mye_l6_on, mye_clus)
data_mye_l6_pre = clean_and_merge_data(adata_mye_l6_pre, mye_clus)


In [28]:
data_mye_g6_on.var_names = data_mye_g6_on.var_names.astype(str)
# print(data_mye_g6_on.var_names)
data_mye_g6_on.var_names = data_mye_g6_on.var_names.astype(str)

data_mye_g6_pre.var_names = data_mye_g6_pre.var_names.astype(str)
# print(data_mye_g6_pre.var_names)
data_mye_g6_pre.var_names = data_mye_g6_pre.var_names.astype(str)

data_mye_l6_on.var_names = data_mye_l6_on.var_names.astype(str)
# print(data_mye_l6_on.var_names)
data_mye_l6_on.var_names = data_mye_l6_on.var_names.astype(str)

data_mye_l6_pre.var_names = data_mye_l6_pre.var_names.astype(str)
# print(data_mye_l6_pre.var_names)
data_mye_l6_pre.var_names = data_mye_l6_pre.var_names.astype(str)

In [29]:
def convert_columns_to_str(adata, columns):
    for col in columns:
        if col in adata.obs.columns:
            adata.obs[col] = adata.obs[col].astype(str)

lst_str_cols = ['orig.ident', 'sampleID', ' specimenID', ' patientID', ' treatment', ' tissueSite', ' response', 
                'MMRstatus',  'pctChangeNoPre', 'category','midcategory','RECIST','sex','resp_tx','phase',
                'resolutionL1', 'cell_type_L1', 'resolution_L2','cell_type_L2','cell_type_L2_long','jon_cell_type']


convert_columns_to_str(data_mye_g6_pre, lst_str_cols)
convert_columns_to_str(data_mye_g6_on, lst_str_cols)
convert_columns_to_str(data_mye_l6_pre, lst_str_cols)
convert_columns_to_str(data_mye_l6_on, lst_str_cols)
mye_l6_pre = data_mye_l6_pre
mye_l6_on = data_mye_l6_on
mye_g6_pre = data_mye_g6_pre
mye_g6_on = data_mye_g6_on

In [84]:
adata_mye_P03_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P03D01']
adata_mye_P03_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P03D15']
adata_mye_P14_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P14D01']
adata_mye_P14_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P14D15']
adata_mye_P15_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P15D01']
adata_mye_P15_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P15D15']
adata_mye_P19_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P19D01']
adata_mye_P19_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P19D15']
adata_mye_P25_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P25D01']
adata_mye_P25_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P25D15']
adata_mye_P27_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P27D01']
adata_mye_P27_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P27D15']
adata_mye_P28_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P28D01']
adata_mye_P28_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P28D15']
adata_mye_P29_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P29D01']
adata_mye_P29_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P29D15']
adata_mye_P30_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P30D01']
adata_mye_P30_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P30D15']
adata_mye_P33_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P33D01']
adata_mye_P33_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P33D15']
adata_mye_P35_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P35D01']
adata_mye_P35_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P35D15']
adata_mye_P36_pre = mye_l6_pre[mye_l6_pre.obs['specimenID'] == 'P36D01']
adata_mye_P36_on = mye_l6_on[mye_l6_on.obs['specimenID'] == 'P36D15']

adata_mye_P02_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P02D01']
adata_mye_P02_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P02D15']
adata_mye_P10_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P10D01']
adata_mye_P10_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P10D15']
adata_mye_P11_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P11D01']
adata_mye_P11_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P11D15']
adata_mye_P13_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P13D01']
adata_mye_P13_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P13D15']
adata_mye_P17_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P17D01']
adata_mye_P17_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P17D15']
adata_mye_P18_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P18D01']
adata_mye_P18_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P18D15']
adata_mye_P20_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P20D01']
adata_mye_P20_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P20D15']
adata_mye_P23_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P23D01']
adata_mye_P23_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P23D15']
adata_mye_P24_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P24D01']
adata_mye_P24_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P24D15']
adata_mye_P26_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P26D01']
adata_mye_P26_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P26D15']
adata_mye_P32_pre = mye_g6_pre[mye_g6_pre.obs['specimenID'] == 'P32D01']
adata_mye_P32_on = mye_g6_on[mye_g6_on.obs['specimenID'] == 'P32D15']
pre_mye_objects = [adata_mye_P03_pre, adata_mye_P14_pre, adata_mye_P15_pre, adata_mye_P19_pre,
               adata_mye_P25_pre, adata_mye_P27_pre, adata_mye_P28_pre, adata_mye_P29_pre,
               adata_mye_P30_pre, adata_mye_P33_pre, adata_mye_P35_pre, adata_mye_P36_pre,
               adata_mye_P02_pre, adata_mye_P10_pre, adata_mye_P11_pre, adata_mye_P13_pre,
               adata_mye_P17_pre, adata_mye_P18_pre, adata_mye_P20_pre, adata_mye_P23_pre,
               adata_mye_P24_pre, adata_mye_P26_pre, adata_mye_P32_pre]
print(adata_mye_P03_pre)
on_mye_objects = [adata_mye_P03_on, adata_mye_P14_on, adata_mye_P15_on, adata_mye_P19_on,
               adata_mye_P25_on, adata_mye_P27_on, adata_mye_P28_on, adata_mye_P29_on,
               adata_mye_P30_on, adata_mye_P33_on, adata_mye_P35_on, adata_mye_P36_on,
               adata_mye_P02_on, adata_mye_P10_on, adata_mye_P11_on, adata_mye_P13_on,
               adata_mye_P17_on, adata_mye_P18_on, adata_mye_P20_on, adata_mye_P23_on,
               adata_mye_P24_on, adata_mye_P26_on, adata_mye_P32_on]

View of AnnData object with n_obs × n_vars = 42 × 26327
    obs: 'cell_id', 'idx', 'sampleID', 'specimenID', 'patientID', 'treatment', 'tissueSite', 'response', 'pctChange', 'MMRstatus', 'pctChangeNoPre', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'leiden', 'doublet_score', 'predicted_doublet', 'category', 'midcategory', 'RECIST', 'PFSmo', 'sex', 'resp_tx', 'S.Score', 'G2M.Score', 'Phase', 'ccDiff.Score', 'L1_RNA_snn_res.0.01_leiden', 'L1_RNA_snn_res.0.02_leiden', 'L1_RNA_snn_res.0.04_leiden', 'L1_RNA_snn_res.0.08_leiden', 'L1_RNA_snn_res.0.16_leiden', 'L1_RNA_snn_res.0.32_leiden', 'L1_RNA_snn_res.0.64_leiden', 'L1_RNA_snn_res.1.28_leiden', 'L1_RNA_snn_res.2.56_leiden', 'L2_RNA_snn_res.0.01_leiden', 'L2_RNA_snn_res.0.02_leiden', 'L2_RNA_snn_res.0.04_leiden', 'L2_RNA_snn_res.0.08_leiden', 'L2_RNA_snn_res.0.16_leiden', 'L2_RNA_snn_res.0.32_leiden', 'L2_humap_res', 'HUMAP_1', 'HUMAP_2', 'L2_humap_fgraph_res.0.4'
    var: 'gene_ids', 'feature_types'

In [96]:
def merge_anndata_objects(adata_list):
    return an.concat(adata_list)

merged_pre_mye = merge_anndata_objects(pre_mye_objects)
merged_on_mye = merge_anndata_objects(on_mye_objects)

# Print the shapes of the merged AnnData objects
print(f"Merged pre-mye shape: {merged_pre_mye.shape}")
print(f"Merged on-mye shape: {merged_on_mye.shape}")

all_mye = an.concat([merged_pre_mye, merged_on_mye])
print(all_mye)

/home/jupyter/.local/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/jupyter/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/jupyter/.local/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/jupyter/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Merged pre-mye shape: (22611, 26327)
Merged on-mye shape: (13670, 26327)
AnnData object with n_obs × n_vars = 36281 × 26327
    obs: 'cell_id', 'idx', 'sampleID', 'specimenID', 'patientID', 'treatment', 'tissueSite', 'response', 'pctChange', 'MMRstatus', 'pctChangeNoPre', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'leiden', 'doublet_score', 'predicted_doublet', 'category', 'midcategory', 'RECIST', 'PFSmo', 'sex', 'resp_tx', 'S.Score', 'G2M.Score', 'Phase', 'ccDiff.Score', 'L1_RNA_snn_res.0.01_leiden', 'L1_RNA_snn_res.0.02_leiden', 'L1_RNA_snn_res.0.04_leiden', 'L1_RNA_snn_res.0.08_leiden', 'L1_RNA_snn_res.0.16_leiden', 'L1_RNA_snn_res.0.32_leiden', 'L1_RNA_snn_res.0.64_leiden', 'L1_RNA_snn_res.1.28_leiden', 'L1_RNA_snn_res.2.56_leiden', 'L2_RNA_snn_res.0.01_leiden', 'L2_RNA_snn_res.0.02_leiden', 'L2_RNA_snn_res.0.04_leiden', 'L2_RNA_snn_res.0.08_leiden', 'L2_RNA_snn_res.0.16_leiden', 'L2_RNA_snn_res.0.32_leiden', 'L2_humap_res', 'HUMAP_1', 'HUMA

/home/jupyter/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [97]:
if 'L2_humap_fgraph_res.0.4' in all_mye.obs.columns:
    all_mye.obs['L2_humap_fgraph_res.0.4'] = all_mye.obs['L2_humap_fgraph_res.0.4'].astype('category')
    all_mye.obs['L2_humap_fgraph_res.0.4'] = all_mye.obs['L2_humap_fgraph_res.0.4'].cat.rename_categories(lambda x: str(x))

# Verify the change
print(all_mye.obs['L2_humap_fgraph_res.0.4'].dtype)
print(all_mye.obs['L2_humap_fgraph_res.0.4'].cat.categories)

print(all_mye)

all_mye = all_mye[~all_mye.obs['L2_humap_fgraph_res.0.4'].isin(['T cell contamination', 'epithelial contamination'])]

# Print the shape of the resulting AnnData object to verify the removal
print(all_mye.shape)

# Print the first few rows of the filtered AnnData object
print(all_mye)

category
Index(['APOC1 Macrophages', 'CXCL10 macrophages', 'DC1', 'DC2',
       'FCN1 monocytes', 'ISGs', 'MHCII', 'Macrophages', 'SPP1 myeloid',
       'mreg DCs', 'pDCs', 'proliferating'],
      dtype='object')
AnnData object with n_obs × n_vars = 36281 × 26327
    obs: 'cell_id', 'idx', 'sampleID', 'specimenID', 'patientID', 'treatment', 'tissueSite', 'response', 'pctChange', 'MMRstatus', 'pctChangeNoPre', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'leiden', 'doublet_score', 'predicted_doublet', 'category', 'midcategory', 'RECIST', 'PFSmo', 'sex', 'resp_tx', 'S.Score', 'G2M.Score', 'Phase', 'ccDiff.Score', 'L1_RNA_snn_res.0.01_leiden', 'L1_RNA_snn_res.0.02_leiden', 'L1_RNA_snn_res.0.04_leiden', 'L1_RNA_snn_res.0.08_leiden', 'L1_RNA_snn_res.0.16_leiden', 'L1_RNA_snn_res.0.32_leiden', 'L1_RNA_snn_res.0.64_leiden', 'L1_RNA_snn_res.1.28_leiden', 'L1_RNA_snn_res.2.56_leiden', 'L2_RNA_snn_res.0.01_leiden', 'L2_RNA_snn_res.0.02_leiden', 'L2_RNA_snn_

In [99]:
\

KeyError: "['FCN1 monocytes'] not in index"